# SwinJSCC on DIV2K → Kodak (150 epochs/channel)

This private Kaggle GPU notebook trains the official SwinJSCC architecture from the author repository on the attached DIV2K training images, then tests all 24 Kodak images under AWGN and Rayleigh at 10 dB. It records the checked-out commit, epoch history, checkpoint, PSNR, SSIM, MS-SSIM, CBR, complex channel-use count, and profiler FLOPs.

**Reproduction scope:** this uses the official `SwinJSCC_w/o_SAandRA` model code, fixed rate C=96 and fixed SNR 10 dB, freshly initialized weights, 256×256 random DIV2K crops, batch size 16, mixed precision, two-GPU DataParallel for fixed-size batches, and 150 epochs per channel. The notebook prints batch-level heartbeats and saves each completed epoch. The official repository hardcodes a workstation dataset path and an effectively unbounded epoch count; those are adapted here. Its Rayleigh layer is its own fading implementation; cross-model Rayleigh results need the channel-model caveat in the local audit.

Paper: [SwinJSCC (IEEE TCCN 2024)](https://arxiv.org/abs/2308.09361) · Official code: [semcomm/swinjscc](https://github.com/semcomm/swinjscc)


In [ ]:
import sys, time
try:
    sys.stdout.reconfigure(line_buffering=True, write_through=True)
except Exception:
    pass
print('KAGGLE BOOT: notebook execution started at', time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()), flush=True)

In [ ]:
from pathlib import Path
import os, subprocess, sys, json, time, random, csv, math
print('STARTUP: standard library loaded; importing NumPy and Pillow...', flush=True)
import numpy as np
from PIL import Image
print('STARTUP: importing PyTorch...', flush=True)
import torch
print('STARTUP: PyTorch import complete.', flush=True)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from types import SimpleNamespace

print('STARTUP: checking CUDA and visible GPUs...', flush=True)
assert torch.cuda.is_available(), 'Enable Kaggle GPU (T4 ×2). '
print('GPU(s):', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], flush=True)
try:
    subprocess.run(['nvidia-smi'], check=False, timeout=10)
except subprocess.TimeoutExpired:
    print('nvidia-smi timed out after 10 seconds; continuing without it.', flush=True)
print('STARTUP: checking mounted datasets...', flush=True)
DIV2K = Path('/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images')
KODAK = Path('/kaggle/input/datasets/sherylmehta/kodak-dataset')
assert DIV2K.exists(), DIV2K
KODAK_FILES = sorted(KODAK.rglob('kodim*.png'))
TRAIN_FILES = sorted(p for p in DIV2K.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'})
assert len(TRAIN_FILES)==800, f'Expected 800 DIV2K images, got {len(TRAIN_FILES)}'
assert len(KODAK_FILES)==24, f'Expected 24 Kodak images, got {len(KODAK_FILES)}'
print('DIV2K images:',len(TRAIN_FILES),'Kodak images:',len(KODAK_FILES), flush=True)
SRC=Path('/kaggle/working/swinjscc')
if not SRC.exists():
    print('STARTUP: cloning official SwinJSCC repository (300 second timeout)...', flush=True)
    subprocess.run(['git','clone','--depth','1','https://github.com/semcomm/swinjscc.git',str(SRC)],check=True,timeout=300)
    print('STARTUP: repository clone complete.', flush=True)
REV=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()
print('Official SwinJSCC commit:',REV,flush=True)
sys.path.insert(0,str(SRC))
from net.network import SwinJSCC
print('STARTUP: installing pytorch-msssim (300 second timeout)...', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pytorch-msssim'],check=True,timeout=300)
print('STARTUP: metric dependency ready.', flush=True)
from pytorch_msssim import ssim, ms_ssim
OUT=Path('/kaggle/working/seven_model_results/SwinJSCC'); OUT.mkdir(parents=True,exist_ok=True)
SEED=20261018; EPOCHS=150; CROP=256; BATCH=16; SNR_DB=10; RATE=96
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

class Div2kRandomCrop(Dataset):
    def __len__(self): return len(TRAIN_FILES)
    def __getitem__(self,i):
        im=Image.open(TRAIN_FILES[i]).convert('RGB'); w,h=im.size
        if w<CROP or h<CROP: im=im.resize((max(w,CROP),max(h,CROP)),Image.Resampling.BICUBIC); w,h=im.size
        x=random.randrange(w-CROP+1); y=random.randrange(h-CROP+1)
        im=im.crop((x,y,x+CROP,y+CROP))
        if random.random()<.5: im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        return torch.from_numpy(np.asarray(im,dtype=np.float32).transpose(2,0,1).copy()/255.)

class DataParallelSafeSwin(nn.Module):
    def __init__(self, core): super().__init__(); self.core=core
    def forward(self,x):
        rec,cbr,snr,mse,loss=self.core(x)
        return rec,torch.tensor(float(cbr),device=x.device),torch.as_tensor(snr,device=x.device,dtype=torch.float32),mse,loss

def build_model(channel):
    args=SimpleNamespace(distortion_metric='MSE',trainset='DIV2K',model='SwinJSCC_w/o_SAandRA',channel_type=channel.lower(),multiple_snr=str(SNR_DB),C=str(RATE),logger=None)
    cfg=SimpleNamespace(pass_channel=True,CUDA=True,device=torch.device('cuda:0'),norm=False,downsample=4,logger=None)
    enc_common=dict(model=args.model,img_size=(CROP,CROP),patch_size=2,in_chans=3,window_size=8,mlp_ratio=4.,qkv_bias=True,qk_scale=None,norm_layer=nn.LayerNorm,patch_norm=True)
    dec_common=dict(model=args.model,img_size=(CROP,CROP),window_size=8,mlp_ratio=4.,qkv_bias=True,qk_scale=None,norm_layer=nn.LayerNorm,patch_norm=True)
    cfg.encoder_kwargs=dict(enc_common,embed_dims=[128,192,256,320],depths=[2,2,6,2],num_heads=[4,6,8,10],C=RATE)
    cfg.decoder_kwargs=dict(dec_common,embed_dims=[320,256,192,128],depths=[2,6,2,2],num_heads=[10,8,6,4],C=RATE)
    core=SwinJSCC(args,cfg).to('cuda:0')
    # Fixed 256x256 training crops keep the upstream resolution cache stable.
    wrapped=DataParallelSafeSwin(core)
    if torch.cuda.device_count()>1:
        print('Using DataParallel on',torch.cuda.device_count(),'GPUs for fixed-size 256x256 batches.',flush=True)
        wrapped=nn.DataParallel(wrapped,device_ids=list(range(torch.cuda.device_count())))
    else:
        print('Using one visible GPU.',flush=True)
    return wrapped

loader=DataLoader(Div2kRandomCrop(),batch_size=BATCH,shuffle=True,num_workers=4,pin_memory=True,drop_last=True,persistent_workers=True)
print('Samples per epoch:',len(loader)*BATCH,'epochs/channel:',EPOCHS,'crop:',CROP,'batch:',BATCH)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def count_flops(model, channel):
    print(f'{channel}: starting FLOPs profiling; this may take a few minutes.', flush=True)
    x=torch.rand(1,3,CROP,CROP,device='cuda:0')
    model.train(); model.zero_grad(set_to_none=True)
    with profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA],with_flops=True) as prof:
        rec,_,_,_,loss=model(x); loss.mean().backward()
    train_flops=sum(int(e.flops or 0) for e in prof.key_averages())
    model.zero_grad(set_to_none=True); model.eval()
    with torch.no_grad(),profile(activities=[ProfilerActivity.CPU,ProfilerActivity.CUDA],with_flops=True) as prof: model(x)
    infer_flops=sum(int(e.flops or 0) for e in prof.key_averages())
    torch.cuda.synchronize(); model.train()
    print(f'{channel}: FLOPs profiling complete. train={train_flops}, inference={infer_flops}', flush=True)
    return train_flops,infer_flops

def run_channel(channel):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED); torch.cuda.manual_seed_all(SEED)
    run=OUT/channel; run.mkdir(parents=True,exist_ok=True)
    print(f'{channel}: constructing model...', flush=True)
    model=build_model(channel); optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)
    print(f'{channel}: model ready.', flush=True)
    train_flops,infer_flops=count_flops(model,channel)
    rows=[]; scaler=torch.amp.GradScaler('cuda')
    for epoch in range(1,EPOCHS+1):
        model.train(); t0=time.time(); total=0.; count=0
        print(f'{channel} epoch {epoch}/{EPOCHS} starting; batches={len(loader)}',flush=True)
        for batch_idx,x in enumerate(loader,1):
            x=x.to('cuda:0',non_blocking=True); optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                _,_,_,_,loss=model(x); loss=loss.mean()
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total+=float(loss.detach())*x.shape[0]; count+=x.shape[0]
            if batch_idx==1 or batch_idx%5==0 or batch_idx==len(loader):
                print(f'{channel} epoch {epoch}/{EPOCHS} batch {batch_idx}/{len(loader)} loss={float(loss.detach()):.6f} elapsed={time.time()-t0:.1f}s',flush=True)
        row={'model':'SwinJSCC','channel':channel,'snr_db':SNR_DB,'epoch':epoch,'train_loss':total/max(count,1),'epoch_seconds':time.time()-t0,'samples_seen':count,'div2k_pool_images':len(TRAIN_FILES),'crop_px':CROP,'batch_size':BATCH,'rate_C':RATE,'nominal_cbr_target':RATE/(2*3*(2**(4*2))),'training_flops_per_sample':train_flops,'inference_flops_per_image_256':infer_flops,'source_commit':REV}
        rows.append(row)
        import pandas as pd
        pd.DataFrame(rows).to_csv(run/'epoch_metrics.csv',index=False)
        torch.save({'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),'config':row},run/'model.pth')
        print('EPOCH COMPLETE',json.dumps(row),flush=True)
    model.eval(); per=[]
    with torch.no_grad():
        for path in KODAK_FILES:
            im=Image.open(path).convert('RGB'); a=np.asarray(im,dtype=np.float32)/255.; h,w=a.shape[:2]
            h0=h-h%128; w0=w-w%128; a=a[:h0,:w0,:]
            ref=torch.from_numpy(a.transpose(2,0,1).copy()).unsqueeze(0).to('cuda:0')
            rec,cbr,_,_,_=model(ref); rec=rec.clamp(0,1)
            mse=F.mse_loss(rec,ref).item(); psnr=10*math.log10(1/max(mse,1e-12))
            s=ssim(rec,ref,data_range=1.,size_average=True).item(); ms=ms_ssim(rec,ref,data_range=1.,size_average=True).item()
            use=int(round(float(cbr.mean())*3*h0*w0)); actual_cbr=use/(3*h0*w0)
            per.append({'model':'SwinJSCC','channel':channel,'snr_db':SNR_DB,'image':path.stem,'height':h0,'width':w0,'psnr_db':psnr,'ssim':s,'msssim':ms,'complex_channel_uses':use,'cbr':actual_cbr,'inference_flops_per_image':infer_flops})
    pd.DataFrame(per).to_csv(run/'kodak_per_image.csv',index=False)
    df=pd.DataFrame(per); summary={'model':'SwinJSCC','channel':channel,'snr_db':SNR_DB,'epochs':EPOCHS,'kodak_psnr_db':df.psnr_db.mean(),'kodak_ssim':df.ssim.mean(),'kodak_msssim':df.msssim.mean(),'mean_complex_channel_uses_per_image':df.complex_channel_uses.mean(),'mean_cbr':df.cbr.mean(),'training_flops_per_sample':train_flops,'inference_flops_per_image_256':infer_flops,'div2k_pool_images':len(TRAIN_FILES),'div2k_images_per_epoch':len(loader)*BATCH,'crop_px':CROP,'batch_size':BATCH,'source_commit':REV}
    pd.DataFrame([summary]).to_csv(run/'summary.csv',index=False); print('CHANNEL SUMMARY',json.dumps(summary),flush=True)
    del model,optimizer; torch.cuda.empty_cache()

for channel in ['AWGN','Rayleigh']: run_channel(channel)
print('All SwinJSCC channels finished. Artifacts:',OUT)
